In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import matplotlib as mpl
import warnings; warnings.simplefilter('ignore')
import os
import sys
import h5py
import pandas as pd
import seaborn as sns
sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
import jsm_ancillary
import jsm_stellarhalo
import jsm_SHMR
import jsm_mcmc
import jsm_stats
import jsm_simload
import evolve as ev
import galhalo as gh
import profiles as profiles
import config as cfg

In [21]:
tree_i = jsm_stellarhalo.Tree_Reader(file="../../data/tree_13.12_588_evo.npz", mass_threshold=7.75e10, verbose=True)
tree_i.compute_concentration(rng=np.random.default_rng(42), c_true="zhao")
# return tree_i.write_out_abundance()

reading in the tree!
converting cyldrical coordinates to cartesian!
counting subhalos in different regimes
saving the subhalo mass function
The total number of surviving (k=1) subhalos: 1
The TRUE value of concentration: 7.319
The concentration with no substructure and a well defined center of mass: 7.241
The total fraction of mass in surviving (k=1) subhalos: 0.194
The concentration with substructure and the same center of mass as before: 10.638
The concentration with substructure and with a new center of mass: 11.061


In [17]:
test = np.array([113.07532615,227.11043748, -43.90029738])

In [19]:
test[:, None]

array([[113.07532615],
       [227.11043748],
       [-43.90029738]])

In [12]:
tree_i.with_sub[-1]

257.47313705860233

In [13]:
tree_i.VirialRadius[0,0]

712.11005

In [ ]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0 #inches
single_textwidth = 3.5 #inches
levelz = [1-0.99, 1-0.95, 1-0.68]

In [ ]:
Npart4 = jsm_ancillary.load_sample("../../data/zhao/ctest_free/concentration_test_Npart4.h5")
Npart5 = jsm_ancillary.load_sample("../../data/zhao/ctest_free/concentration_test_Npart5.h5")
Npart6 = jsm_ancillary.load_sample("../../data/zhao/ctest_free/concentration_test_Npart6.h5")
Npart7 = jsm_ancillary.load_sample("../../data/zhao/ctest_free/concentration_test_Npart7.h5")

In [ ]:
def summary_stats(df):

    real_cvirs = jsm_ancillary.make_matrix(df, "host_c")[:, 0]
    smooth = jsm_ancillary.make_matrix(df, "c_measured_smooth")[:, 0]
    fixed_COM = jsm_ancillary.make_matrix(df, "c_measured_fixed_COM")[:, 0]
    shifted_COM = jsm_ancillary.make_matrix(df, "c_measured_shifted_COM")[:, 0]
    z50s = np.log10(1 + df["host_z50"])
    fsub = np.log10(df["fsub_used"])
    Nsub = np.log10(df["Nsub_used"])

    return np.array([real_cvirs, smooth, fixed_COM, shifted_COM]), z50s.values, fsub.values, Nsub.values

In [ ]:
c4s, z50s7, fsub4, Nsub4 = summary_stats(Npart4)

c5s, z50s7, fsub5, Nsub5 = summary_stats(Npart5)

c6s, z50s7, fsub6, Nsub6 = summary_stats(Npart6)

c7s, z50s7, fsub7, Nsub7 = summary_stats(Npart7)

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(double_textwidth, double_textwidth),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

xmin = 4
xmax = 35

# --------------------------------------------------
# Top panel: recovered vs. true concentration
# --------------------------------------------------

ax = axes[0]

ax.plot([xmin, xmax], [xmin, xmax], "k--", lw=1, zorder=0)

ax.scatter(c4s[0, :], c4s[1, :], marker=".", zorder=1,
           label="N$_{\\rm particles}$ = 1e4", color="C3", s=2)
ax.scatter(c5s[0, :], c5s[1, :], marker=".", zorder=2,
           label="N$_{\\rm particles}$ = 1e5", color="C2", s=2)
ax.scatter(c6s[0, :], c6s[1, :], marker=".", zorder=3,
           label="N$_{\\rm particles}$ = 1e6", color="C1", s=2)
ax.scatter(c7s[0, :], c7s[1, :], marker=".", zorder=4,
           label="N$_{\\rm particles}$ = 1e7", color="C0")

ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
ax.set_ylabel("Recovered concentration")
ax.legend()

# --------------------------------------------------
# Bottom panel: residuals
# --------------------------------------------------

ax = axes[1]

ax.axhline(0, color="k", linestyle="--", lw=1, zorder=0)

ax.scatter(c4s[0, :], c4s[0, :] - c4s[1, :], marker=".", zorder=1, color="C3", s=2)
ax.scatter(c5s[0, :], c5s[0, :] - c5s[1, :], marker=".", zorder=2, color="C2", s=2)
ax.scatter(c6s[0, :], c6s[0, :] - c6s[1, :], marker=".", zorder=3, color="C1", s=2)
ax.scatter(c7s[0, :], c7s[0, :] - c7s[1, :], marker=".", zorder=4, color="C0")

ax.set_xlim(xmin, xmax)
ax.set_xlabel("True concentration")
ax.set_ylabel("Δc")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(double_textwidth, double_textwidth),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

xmin = 4
xmax = 35

# --------------------------------------------------
# Top panel: recovered vs. true concentration
# --------------------------------------------------

ax = axes[0]

ax.plot([xmin, xmax], [xmin, xmax], "k--", lw=1, zorder=0)

ax.scatter(c4s[0, :], c4s[2, :], marker=".", zorder=1,
           label="N$_{\\rm particles}$ = 1e4", color="C3", s=2)
ax.scatter(c5s[0, :], c5s[2, :], marker=".", zorder=2,
           label="N$_{\\rm particles}$ = 1e5", color="C2", s=2)
ax.scatter(c6s[0, :], c6s[2, :], marker=".", zorder=3,
           label="N$_{\\rm particles}$ = 1e6", color="C1", s=2)
ax.scatter(c7s[0, :], c7s[2, :], marker=".", zorder=4,
           label="N$_{\\rm particles}$ = 1e7", color="C0")

ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
ax.set_ylabel("Recovered concentration")
ax.legend()

# --------------------------------------------------
# Bottom panel: residuals
# --------------------------------------------------

ax = axes[1]

ax.axhline(0, color="k", linestyle="--", lw=1, zorder=0)

ax.scatter(c4s[0, :], c4s[0, :] - c4s[2, :], marker=".", zorder=1, color="C3", s=2)
ax.scatter(c5s[0, :], c5s[0, :] - c5s[2, :], marker=".", zorder=2, color="C2", s=2)
ax.scatter(c6s[0, :], c6s[0, :] - c6s[2, :], marker=".", zorder=3, color="C1", s=2)
ax.scatter(c7s[0, :], c7s[0, :] - c7s[2, :], marker=".", zorder=4, color="C0")

ax.set_xlim(xmin, xmax)
ax.set_xlabel("True concentration")
ax.set_ylabel("Δc")
ax.set_ylim(-40, 20)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(double_textwidth, double_textwidth),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

xmin = 4
xmax = 35

# --------------------------------------------------
# Top panel: recovered vs. true concentration
# --------------------------------------------------

ax = axes[0]

ax.plot([xmin, xmax], [xmin, xmax], "k--", lw=1, zorder=0)

ax.scatter(c4s[0, :], c4s[3, :], marker=".", zorder=1,
           label="N$_{\\rm particles}$ = 1e4", color="C3", s=2)
ax.scatter(c5s[0, :], c5s[3, :], marker=".", zorder=2,
           label="N$_{\\rm particles}$ = 1e5", color="C2", s=2)
ax.scatter(c6s[0, :], c6s[3, :], marker=".", zorder=3,
           label="N$_{\\rm particles}$ = 1e6", color="C1", s=2)
ax.scatter(c7s[0, :], c7s[3, :], marker=".", zorder=4,
           label="N$_{\\rm particles}$ = 1e7", color="C0")

ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
ax.set_ylabel("Recovered concentration")
ax.legend()

# --------------------------------------------------
# Bottom panel: residuals
# --------------------------------------------------

ax = axes[1]

ax.axhline(0, color="k", linestyle="--", lw=1, zorder=0)

ax.scatter(c4s[0, :], c4s[0, :] - c4s[3, :], marker=".", zorder=1, color="C3", s=2)
ax.scatter(c5s[0, :], c5s[0, :] - c5s[3, :], marker=".", zorder=2, color="C2", s=2)
ax.scatter(c6s[0, :], c6s[0, :] - c6s[3, :], marker=".", zorder=3, color="C1", s=2)
ax.scatter(c7s[0, :], c7s[0, :] - c7s[3, :], marker=".", zorder=4, color="C0")

ax.set_xlim(xmin, xmax)
ax.set_xlabel("True concentration")
ax.set_ylabel("Δc")
ax.set_ylim(-40, 20)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr

def plotkde_and_measurerho(xvals, yvals, xlim, ylim, xlabel, ylabel, color="C0"):

    mask = np.isfinite(xvals) & np.isfinite(yvals)
    x = xvals[mask]
    y = yvals[mask]

    sns.kdeplot(x=x, y=y, levels=levelz, color=color)

    rho, p = spearmanr(x, y)

    plt.text(0.05, 0.15,
             f"$\\rho = {rho:.3f}$",
             transform=plt.gca().transAxes,
             ha="left", va="top")

    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [ ]:
plotkde_and_measurerho(np.log10(c7s[0, :]), fsub7, (0.5, 1.6), (-3, 0), "log c$_{\\rm vir}$ (Zhao et al. 2009)", "log f$_{\\rm sub}$")

In [ ]:
plotkde_and_measurerho(np.log10(c7s[2, :]), fsub7, (0.5, 1.6), (-3, 0), "log c$_{\\rm vir}$ (fixed COM)", "log f$_{\\rm sub}$")

In [ ]:
plotkde_and_measurerho(np.log10(c7s[3, :]), fsub7, (0.5, 1.6), (-3, 0), "log c$_{\\rm vir}$ (shifted COM)", "log f$_{\\rm sub}$")

In [ ]:
plotkde_and_measurerho(np.log10(c4s[3, :]), fsub4, (0.5, 1.6), (-3, 0), "log c$_{\\rm vir}$ (shifted COM)", "log f$_{\\rm sub}$", color="C3")

In [ ]:
plotkde_and_measurerho(np.log10(c7s[0, :]), Nsub7, (0.5, 1.6), (0,1.5), "log c$_{\\rm vir}$ (Zhao et al. 2009)", "log N$_{\\rm sub}$")

In [ ]:
plotkde_and_measurerho(np.log10(c7s[2, :]), Nsub7, (0.5, 1.6), (0,1.5), "log c$_{\\rm vir}$ (fixed COM)", "log N$_{\\rm sub}$")

In [ ]:
plotkde_and_measurerho(np.log10(c7s[3, :]), Nsub7, (0.5, 1.6), (0,1.5), "log c$_{\\rm vir}$ (shifted COM)", "log N$_{\\rm sub}$")

In [ ]:
plotkde_and_measurerho(fsub7, Nsub7, (-3, 0), (0,1.5), "log f$_{\\rm sub}$", "log N$_{\\rm sub}$")

In [ ]:
masscuts = [5e9, 1e10, 5e10, 1e11]


In [ ]:
masscuts